In [ ]:
# Hosted D2L setup: fetch the exact helper module used to build this notebook.
from pathlib import Path
from urllib.request import urlretrieve
from importlib.metadata import PackageNotFoundError, version
import importlib.util, os, subprocess, sys

required = ['numpy', 'pandas', 'matplotlib', 'requests', 'scipy', 'pillow', 'regex', 'torch', 'torchvision']
imports = {'pillow': 'PIL'}
pinned = {}
fallbacks = {'torch': 'torch==2.11.0', 'torchvision': 'torchvision==0.26.0'}
device = os.environ.get("D2L_HOSTED_DEVICE", "auto").lower()
if device not in ("auto", "cpu", "gpu"):
    raise ValueError(f"Invalid D2L_HOSTED_DEVICE={device!r}")
if device == "auto":
    try:
        gpu = (Path("/dev/nvidia0").exists() or
               subprocess.run(["nvidia-smi", "-L"], capture_output=True,
                              timeout=5).returncode == 0)
    except (FileNotFoundError, subprocess.SubprocessError):
        gpu = False
else:
    gpu = device == "gpu"
if not gpu:
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")
    os.environ.setdefault("JAX_PLATFORMS", "cpu")
tensorflow_version = None
if 'pytorch' in ("tensorflow", "jax"):
    try:
        tensorflow_version = version("tensorflow")
    except PackageNotFoundError:
        pass
# Colab's CPU image currently carries a CUDA-enabled TensorFlow wheel. Its
# first ordinary tensor operation probes CUDA and emits an error-level cuInit
# diagnostic. JAX notebooks also use TensorFlow for data loading, so overlay
# the matching CPU build in both CPU variants. Keep the provider's
# ``tensorflow`` distribution metadata: other preinstalled Colab packages
# depend on that distribution name, while both wheels expose the same module.
if not gpu and 'pytorch' in ("tensorflow", "jax"):
    try:
        tensorflow_cpu_version = version("tensorflow-cpu")
    except PackageNotFoundError:
        tensorflow_cpu_version = None
    if (tensorflow_version is not None and
            tensorflow_cpu_version != tensorflow_version):
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "--no-deps",
            f"tensorflow-cpu=={tensorflow_version}",
        ])
if "tf-keras" in fallbacks and tensorflow_version is not None:
    fallbacks["tf-keras"] = f"tf-keras=={tensorflow_version}"
missing = []
for package in required:
    if package in pinned:
        wanted, cpu_requirement, gpu_requirement, match = pinned[package]
        requirement = gpu_requirement if gpu else cpu_requirement
        if requirement is None:  # a GPU-only package on a CPU runtime
            continue
        try:
            installed = version(package)
        except PackageNotFoundError:
            installed = None
        actual = (installed.split("+", 1)[0]
                  if installed is not None and match == "public" else installed)
        if actual != wanted:
            missing.append(requirement)
    elif importlib.util.find_spec(imports.get(package, package)) is None:
        missing.append(fallbacks.get(package, package))
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

mismatched = []
for package, (wanted, cpu_requirement, _, match) in pinned.items():
    if not gpu and cpu_requirement is None:
        continue
    try:
        installed = version(package)
    except PackageNotFoundError:
        installed = None
    actual = (installed.split("+", 1)[0]
              if installed is not None and match == "public" else installed)
    if actual != wanted:
        mismatched.append(f"{package}={installed!r} (expected {wanted})")
if mismatched:
    raise RuntimeError("Hosted runtime setup failed: " + ", ".join(mismatched))

root = Path(".d2l-hosted") / "85bd815efd86af24338a0f0c45844cb7e1e6f155"
package = root / "d2l"
package.mkdir(parents=True, exist_ok=True)
base = "https://raw.githubusercontent.com/smolix/d2l-neu/85bd815efd86af24338a0f0c45844cb7e1e6f155/d2l"
for name in ('__init__.py', 'torch.py'):
    target = package / name
    if not target.exists():
        urlretrieve(f"{base}/{name}", target)
if str(root.resolve()) not in sys.path:
    sys.path.insert(0, str(root.resolve()))
pythonpath = os.environ.get("PYTHONPATH", "").split(os.pathsep)
if str(root.resolve()) not in pythonpath:
    os.environ["PYTHONPATH"] = os.pathsep.join(
        [str(root.resolve()), *[entry for entry in pythonpath if entry]]
    )


# Multiple Input and Multiple Output Channels

So far our convolutions have mapped a single 2D grid to a single 2D grid. Real
images and hidden representations have multiple *channels* on both sides: an RGB
photograph is a $3 \times h \times w$ tensor (requiring cross-channel comparisons
to detect a red-on-green boundary of equal luminance), and a hidden layer
extracts dozens or hundreds of distinct feature maps in parallel.

As derived in that section (the equation), a layer
with $c_\textrm{i}$ input channels and $c_\textrm{o}$ output channels maintains
one $c_\textrm{i} \times k_\textrm{h} \times k_\textrm{w}$ kernel per output
channel, cross-correlating across all input channels and summing the results. In
this section, we implement multi-channel convolutions, account for their
parameter, FLOP, and activation-memory costs (governed by the product
$c_\textrm{i} c_\textrm{o}$), and introduce $1 \times 1$ bottleneck convolutions.

In [ ]:
import math
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from d2l import torch as d2l

## Multiple Input Channels

Let us begin with the input side. When the input contains $c_\textrm{i}$
channels, the kernel must supply one $k_\textrm{h}\times k_\textrm{w}$ filter
for each input channel, giving a third-order kernel tensor of shape
$c_\textrm{i}\times k_\textrm{h}\times k_\textrm{w}$. We cross-correlate
every input channel $\mathbf{X}_c$ with its corresponding kernel slice
$\mathbf{K}_c$ using the two-dimensional cross-correlation the equation
(computed by `corr2d`) and sum the $c_\textrm{i}$ intermediate grids:

$$\mathbf{Y} = \sum_{c=1}^{c_\textrm{i}} \mathbf{X}_c \star \mathbf{K}_c.$$

Each entry of the resulting two-dimensional output is therefore a weighted sum
over the same spatial window across all input channels, combining
$c_\textrm{i} k_\textrm{h} k_\textrm{w}$ scalar products.
the figure illustrates this calculation for two input
channels, where the shaded upper-left output entry equals
$(1\times1+2\times2+4\times3+5\times4)+(0\times0+1\times1+3\times2+4\times3)=56$.

![Cross-correlation with two input channels: one cross-correlation per channel, then a sum.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/conv-multi-in.svg)

Summing across channels is precisely what allows a kernel to compare
information carried by different channels, such as color. Consider a
$1\times 1$ kernel with weight $+1$ on the red channel and $-1$ on the green
channel: at every pixel it computes red minus green, producing positive values
in red regions, negative values in green regions, and zero in neutral gray
regions regardless of brightness. Wherever its sign flips between adjacent
pixels, a red region meets a green one---even if both regions have identical
luminance, in which case a grayscale brightness filter would see no edge at
all.

Implementing multi-input cross-correlation translates the formula directly
into Python: we call `corr2d` once per channel and sum the results.

In [ ]:
def corr2d_multi_in(X, K):
    """Cross-correlate each channel of X with its channel of K and add."""
    return sum(d2l.corr2d(x, k) for x, k in zip(X, K))

Let us verify the implementation on the input and kernel tensors from
the figure; the upper-left output entry should be 56.

In [ ]:
X = d2l.tensor([[[0.0, 1.0, 2.0], [3.0, 4.0, 5.0], [6.0, 7.0, 8.0]],
               [[1.0, 2.0, 3.0], [4.0, 5.0, 6.0], [7.0, 8.0, 9.0]]])
K = d2l.tensor([[[0.0, 1.0], [2.0, 3.0]], [[1.0, 2.0], [3.0, 4.0]]])

corr2d_multi_in(X, K)

## Multiple Output Channels

No matter how many input channels it reads, a single multi-input kernel
produces just one two-dimensional *feature map*
(that section). To extract $c_\textrm{o}$ distinct
feature maps simultaneously, a convolutional layer maintains $c_\textrm{o}$
such kernels, each of shape
$c_\textrm{i}\times k_\textrm{h}\times k_\textrm{w}$. Stacking them along a
new axis yields a fourth-order kernel tensor of shape
$c_\textrm{o}\times c_\textrm{i}\times k_\textrm{h}\times k_\textrm{w}$,
where output channel $o$ is given by

$$\mathbf{Y}_o = \sum_{c=1}^{c_\textrm{i}} \mathbf{X}_c \star \mathbf{K}_{o,c}, \quad o = 1, \ldots, c_\textrm{o}.$$

Notice that every output channel connects to *every* input channel through its
own $k_\textrm{h}\times k_\textrm{w}$ slice $\mathbf{K}_{o,c}$; this dense
all-to-all channel wiring is the primary driver of the layer's computational
cost. Written channel by channel, this matches the equation
from that section. In typical CNN architectures, the number of
channels grows as spatial resolution shrinks via strides or pooling, trading
fine-grained spatial layout for richer, jointly learned feature descriptions
at each location.
<!-- d2l:prose id=channels-md-5 fw=mxnet, pytorch -->

Gluon and PyTorch store kernels channels-first as $(c_\textrm{o}, c_\textrm{i}, k_\textrm{h}, k_\textrm{w})$.

To implement multi-channel output cross-correlation, we iterate over the
output kernels, apply `corr2d_multi_in` to each, and stack the resulting
feature maps along axis 0.

In [ ]:
def corr2d_multi_in_out(X, K):
    """Apply corr2d_multi_in once per output channel of K and stack."""
    return d2l.stack([corr2d_multi_in(X, k) for k in K], 0)

Let us test this function by stacking `K`, `K + 1`, and `K + 2` into a kernel
tensor with three output channels.

In [ ]:
K = d2l.stack((K, K + 1, K + 2), 0)
K.shape

In [ ]:
corr2d_multi_in_out(X, K)

The first output channel matches our earlier result because its kernel is the
original `K`. We can verify the second channel by mental arithmetic: adding 1
to every kernel weight increases each output entry by the sum of all elements
in its receptive window across both input channels. For the upper-left window
that sum is 20, so the second channel begins at $56 + 20 = 76$.

### Parameters and Cost

Adding channels gives a network expressive power, but it also scales up memory
and computation rapidly. Including one scalar bias per output channel, a
convolutional layer contains

$$c_\textrm{o} c_\textrm{i} k_\textrm{h} k_\textrm{w} + c_\textrm{o}$$

learnable parameters. Near the input of a network, this number is modest: the
$3\times 3$ layer mapping 3 RGB channels to 64 channels in
that section has only 1,792 parameters. Deeper inside the network,
however, channel counts grow large: a $3\times 3$ layer mapping 256 channels
to 256 channels requires 589,824 weights plus 256 biases.

Turning to computational cost, each output element requires one multiply-add
for every weight in its output channel---$c_\textrm{i} k_\textrm{h} k_\textrm{w}$
in all. Across an output feature map of spatial size
$h_\textrm{o}\times w_\textrm{o}$ and $c_\textrm{o}$ output channels, the
layer performs

$$h_\textrm{o} w_\textrm{o} \cdot c_\textrm{o} c_\textrm{i} k_\textrm{h} k_\textrm{w}$$

multiply-adds. Applied to a $56\times 56$ feature map, our 256-to-256 layer
executes $56^2\cdot 589{,}824 \approx 1.8\times 10^9$ multiply-adds for a
single image. Both parameter count and arithmetic cost scale with the product
$c_\textrm{i} c_\textrm{o}$, meaning that doubling the width of every layer
quadruples both. Because the same weights are reused at every spatial
location, arithmetic cost also scales directly with the output area
$h_\textrm{o} w_\textrm{o}$.

A third resource cost is just as important during training yet easily
overlooked: *activation memory*. To compute gradients in the backward pass,
the framework must retain each layer's output activations, storing
$h_\textrm{o} w_\textrm{o} c_\textrm{o}$ numbers per training example
(that section). Unlike parameter memory, activation
memory is largest in the *earliest* layers, where spatial resolution is
highest even though parameter counts are tiny. For example, a 3-to-64 layer
producing a $224\times 224$ output must cache
$224^2\cdot 64 \approx 3.2\times 10^6$ activations per example---nearly two
thousand times more numbers than its 1,792 parameters.

## The $1\times 1$ Convolution

At first glance, a $1\times 1$ convolution ($k_\textrm{h} = k_\textrm{w} = 1$)
might look pointless: because its spatial window covers only a single pixel,
it cannot detect edges, corners, or any other pattern spanning neighboring
locations. When multiple channels are present, however, it performs a very
useful operation---just as our red-minus-green color filter did earlier. At
every spatial coordinate $(i, j)$, a $1\times 1$ convolution multiplies the
$c_\textrm{i}$-dimensional channel vector by a shared
$c_\textrm{o}\times c_\textrm{i}$ weight matrix to produce a
$c_\textrm{o}$-dimensional output vector. In other words, it acts as a fully
connected layer applied independently at every pixel with weights tied across
all spatial positions. the figure depicts a $1\times 1$
convolution mapping 3 input channels to 2 output channels while leaving height
and width intact.

![A $1\times 1$ convolution with three input channels and two output channels. Each output element is a linear combination of the input elements at the same position.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/conv-1x1.svg)

Because the exact same linear map is applied at every pixel, we can compute
the entire $1\times 1$ convolution with a single matrix multiplication: we
reshape the input into a $c_\textrm{i}\times hw$ matrix, left-multiply by the
$c_\textrm{o}\times c_\textrm{i}$ kernel matrix, and reshape the output back
to $(c_\textrm{o}, h, w)$.

In [ ]:
def corr2d_multi_in_out_1x1(X, K):
    """A 1x1 convolution as one matrix product."""
    c_i, h, w = X.shape
    c_o = K.shape[0]
    Y = d2l.matmul(d2l.reshape(K, (c_o, c_i)), d2l.reshape(X, (c_i, h * w)))
    return d2l.reshape(Y, (c_o, h, w))

Let us verify on random inputs that this matrix-multiplication formulation
agrees with `corr2d_multi_in_out`.

In [ ]:
X = d2l.normal(0, 1, (3, 3, 3))
K = d2l.normal(0, 1, (2, 3, 1, 1))
Y1 = corr2d_multi_in_out_1x1(X, K)
Y2 = corr2d_multi_in_out(X, K)
float(d2l.reduce_sum(d2l.abs(Y1 - Y2)))

As expected, the two results match up to floating-point rounding error.

Why are $1\times 1$ convolutions so ubiquitous in modern architectures? Their
main strength is adjusting channel dimensionality cheaply so that expensive
spatial convolutions can operate on narrower representations. Return to our
256-to-256 $3\times 3$ layer. Instead of applying $3\times 3$ kernels directly
to all 256 channels, we can use a $1\times 1$ convolution to project 256
channels down to 64, apply the $3\times 3$ convolution across those 64
channels, and use a second $1\times 1$ convolution to expand back to 256.
Together, the three layers use only
$16{,}384 + 36{,}864 + 16{,}384 = 69{,}632$ weights instead of $589{,}824$---a
reduction of roughly $8.5\times$ in both parameters and multiply-adds. Such a
*bottleneck* restricts intermediate spatial mixing to a 64-channel subspace, a
tradeoff that architectures such as ResNet (that section) happily make
in exchange for much greater depth. Placing an activation function after a
$1\times 1$ convolution turns it into a tiny nonlinear MLP evaluated at every
pixel [@Lin.Chen.Yan.2013]; without a nonlinearity, consecutive linear
convolutions would simply collapse into a single linear map.

Two other architectural mechanisms reduce the $c_\textrm{i} c_\textrm{o}$
factor by restricting which input channels feed into each output channel:
*grouped convolutions*, popularized by ResNeXt (that section), and
*depthwise-separable convolutions*, which underpin lightweight mobile models
(that section). We will introduce each in the next
chapter as we build the architectures that rely on them.

Notice that channels alone leave one basic property unchanged: convolutional
outputs still shift pixel-for-pixel whenever the input shifts. In
that section, we introduce pooling layers to summarize local
neighborhoods and build tolerance to small spatial shifts.

## Summary

To handle multi-channel inputs, a convolutional layer cross-correlates each
input channel with its own spatial kernel and sums the results, allowing a
single filter to integrate cues across channels such as color. To produce
$c_\textrm{o}$ output feature maps, the layer stacks $c_\textrm{o}$ such
multi-input kernels into a tensor of shape
$c_\textrm{o}\times c_\textrm{i}\times k_\textrm{h}\times k_\textrm{w}$,
incurring $c_\textrm{o} c_\textrm{i} k_\textrm{h} k_\textrm{w} + c_\textrm{o}$
parameters and
$h_\textrm{o} w_\textrm{o} c_\textrm{o} c_\textrm{i} k_\textrm{h} k_\textrm{w}$
multiply-adds---both proportional to $c_\textrm{i} c_\textrm{o}$---while
training must also store $h_\textrm{o} w_\textrm{o} c_\textrm{o}$ activations
per example. A $1\times 1$ convolution acts as a shared fully connected layer
at every pixel, letting architectures mix channels and construct parameter-
and compute-efficient bottlenecks.

## Exercises

1. **Composition of convolutions.** Two convolutional layers with stride
   1, no padding, and no nonlinearity in between have kernels of shape
   $c_\textrm{m} \times c_\textrm{i} \times k_1 \times k_1$ and
   $c_\textrm{o} \times c_\textrm{m} \times k_2 \times k_2$.
    1. Prove that their composition is a single convolution, and give the
       shape of its kernel.
    1. Compare the number of weights of the pair with that of the single
       convolution. For which intermediate width $c_\textrm{m}$ is the
       pair cheaper?
    1. As that section showed, three stacked single-channel
       $3 \times 3$ kernels have a $7 \times 7$ receptive field. Decide
       whether every single-channel $7 \times 7$ kernel is the composition
       of three $3 \times 3$ kernels, and justify the answer.

1. **Three costs, three layers.** A network for $224\times 224$ RGB
   images begins with four $3\times 3$ convolutions with padding 1: from 3
   to 64 channels at stride 1, from 64 to 128 at stride 2, from 128 to 128
   at stride 1, and from 128 to 256 at stride 2.
    1. For each layer give the output shape, the number of weights, the
       number of multiply-adds per image, and the number of output values
       that training must keep for the backward pass.
    1. Which layer has the most weights, which performs the most
       multiply-adds, and which keeps the most output values? Explain why
       these are three different layers.
    1. The images are replaced by $448\times 448$ ones. Which of the three
       costs change, and by what factor?

1. [code] **Multi-channel im2col.** Extend the patch matrix of
   that section to several input and output channels.
   Rebuild the two-channel `X` of the figure and the
   three-output-channel `K` of that section,
   since later cells of this section overwrite both. Build the patch
   matrix, whose rows are input patches of shape
   $c_\textrm{i} \times k_\textrm{h} \times k_\textrm{w}$ flattened in
   that order, and arrange `K` into a
   $c_\textrm{i} k_\textrm{h} k_\textrm{w} \times c_\textrm{o}$ kernel
   matrix whose rows follow the same order. Confirm that their product,
   rearranged into the shape of the output, reproduces
   `corr2d_multi_in_out(X, K)`.

    *Adapted from Simon Prince,
    [Understanding Deep Learning](https://udlbook.github.io/udlbook/),
    Problems 10.6 and 10.7.*

[Discussions](https://d2l.discourse.group/t/70)